<a href="https://colab.research.google.com/github/avindumihisara0229-code/ErgoSense/blob/Avindu/Posture%20model%203%20(xgboost)/xgboost.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import cv2
import mediapipe as mp
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report
import joblib
from google.colab import drive
from tqdm.notebook import tqdm

drive.mount('/content/drive')
print("✅ Step 1: Setup complete. Libraries installed and Drive mounted.")

# ==============================================================================
# STEP 2: DEFINE FILE PATHS & LABELS
# ==============================================================================
DATASET_PATH = '/content/drive/MyDrive/'

correct_folders = [
    os.path.join(DATASET_PATH, 'old', '0'),
    os.path.join(DATASET_PATH, 'old', '2'),
    os.path.join(DATASET_PATH, 'old', '3')
]

incorrect_folders = [
    os.path.join(DATASET_PATH, 'old', 'bad', '0'),
    os.path.join(DATASET_PATH, 'old', 'bad', '2'),
    os.path.join(DATASET_PATH, 'old', 'bad', '3')
]

# ==============================================================================
# STEP 3: FEATURE EXTRACTION LOGIC
# ==============================================================================
mp_pose = mp.solutions.pose
pose_model = mp_pose.Pose(static_image_mode=True, min_detection_confidence=0.5, min_tracking_confidence=0.5)

def calculate_angle(a, b, c):
    a, b, c = np.array(a), np.array(b), np.array(c)
    radians = np.arctan2(c[1]-b[1], c[0]-b[0]) - np.arctan2(a[1]-b[1], a[0]-b[0])
    angle = np.abs(radians * 180.0 / np.pi)
    if angle > 180.0:
        angle = 360 - angle
    return angle

def extract_posture_landmarks_from_path(pose_estimator, image_path):
    image = cv2.imread(image_path)
    if image is None: return None
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    results = pose_estimator.process(image_rgb)
    if not results.pose_landmarks: return None
    landmarks = results.pose_landmarks.landmark
    try:
        left_shoulder = [landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].x, landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].y]
        left_ear = [landmarks[mp_pose.PoseLandmark.LEFT_EAR.value].x, landmarks[mp_pose.PoseLandmark.LEFT_EAR.value].y]
        left_hip = [landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].x, landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].y]
        left_knee = [landmarks[mp_pose.PoseLandmark.LEFT_KNEE.value].x, landmarks[mp_pose.PoseLandmark.LEFT_KNEE.value].y]
        neck_angle = calculate_angle(left_ear, left_shoulder, left_hip)
        back_angle = calculate_angle(left_shoulder, left_hip, left_knee)
        return [neck_angle, back_angle]
    except:
        return None

# ==============================================================================
# STEP 4: PROCESS THE DATASET
# ==============================================================================
print("\n⏳ Starting feature extraction...")
features, labels = [], []
label_map = {"correct": 0, "incorrect": 1}

for folder_path in correct_folders:
    if not os.path.isdir(folder_path): continue
    for filename in tqdm(os.listdir(folder_path), desc=f"Processing {os.path.basename(folder_path)}"):
        if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
            landmarks = extract_posture_landmarks_from_path(pose_model, os.path.join(folder_path, filename))
            if landmarks:
                features.append(landmarks)
                labels.append(label_map["correct"])

for folder_path in incorrect_folders:
    if not os.path.isdir(folder_path): continue
    for filename in tqdm(os.listdir(folder_path), desc=f"Processing {os.path.basename(folder_path)}"):
        if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
            landmarks = extract_posture_landmarks_from_path(pose_model, os.path.join(folder_path, filename))
            if landmarks:
                features.append(landmarks)
                labels.append(label_map["incorrect"])

# ==============================================================================
# STEP 5: TRAIN XGBOOST
# ==============================================================================
if len(features) < 10:
    print("❌ Not enough data.")
else:
    X, y = np.array(features), np.array(labels)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

    print("\n⏳ Training XGBoost...")
    model = XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=5, random_state=42, use_label_encoder=False, eval_metric='logloss')
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    print(f"\n✅ Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%")

    joblib.dump(model, '/content/drive/MyDrive/posture_model_xgboost.joblib')
    pd.DataFrame(features, columns=['neck', 'back']).assign(label=labels).to_csv('/content/drive/MyDrive/posture_features.csv', index=False)
    print("✅ Model and CSV saved.")

/usr/local/lib/python3.12/dist-packages/jaxlib/plugin_support.py:71: RuntimeWarning: JAX plugin jax_cuda12_plugin version 0.7.2 is installed, but it is not compatible with the installed jaxlib version 0.7.1, so it will not be used.
  warnings.warn(


Mounted at /content/drive
✅ Step 1: Setup complete. Libraries installed and Drive mounted.

⏳ Starting feature extraction...


Processing 0:   0%|          | 0/146 [00:00<?, ?it/s]

Processing 2:   0%|          | 0/145 [00:00<?, ?it/s]

Processing 3:   0%|          | 0/146 [00:00<?, ?it/s]

...

Processing 2:   0%|          | 0/146 [00:00<?, ?it/s]

Processing 3:   0%|          | 0/146 [00:00<?, ?it/s]


⏳ Training XGBoost...

✅ Accuracy: 77.60%


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [08:16:57] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


✅ Model and CSV saved.


In [3]:


    y_pred = model.predict(X_test)
    print(f"\nModel Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%")
    print("\nClassification Report:\n", classification_report(y_test, y_pred, target_names=["correct", "incorrect"]))

    MODEL_SAVE_PATH = '/content/drive/MyDrive/posture_model.joblib'
    joblib.dump(model, MODEL_SAVE_PATH)
    print(f"\n✅🎉 Model saved as 'posture_model.joblib' to your Google Drive!")



Model Accuracy: 77.60%

Classification Report:
               precision    recall  f1-score   support

     correct       0.77      0.77      0.77        60
   incorrect       0.78      0.78      0.78        65

    accuracy                           0.78       125
   macro avg       0.78      0.78      0.78       125
weighted avg       0.78      0.78      0.78       125


✅🎉 Model saved as 'posture_model.joblib' to your Google Drive!
